# 03 – Phase 3: Post-Training Quantization (PTQ)

Pipeline:
1. Install dependencies
2. Export FP32 model → ONNX
3. Static INT8 quantization (with calibration)
4. Dynamic INT8 quantization (no calibration)
5. Benchmark FP32 vs INT8 (latency, FPS, size)
6. Evaluate accuracy (Dice, IoU, per-class)
7. Generate comparison report
8. Visualize results

**Why ONNX Runtime?**  
EfficientNet-B4 uses SiLU activations incompatible with PyTorch's
static quantization. ONNX Runtime handles arbitrary op graphs natively.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q onnx onnxruntime

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/wet-amd-segmentation-edge/project')

import torch
import numpy as np

from utils import Config, set_seed, get_device, device_info
from models.model_loader import load_model
from models.baseline_model import CLASS_NAMES

from quantization import (
    QuantConfig, PTQPipeline,
    export_to_onnx, create_ort_session,
    prepare_model_for_export, optimise_onnx_graph, list_onnx_ops,
    onnx_size_mb, compression_ratio,
    benchmark_ort_session, evaluate_ort_model,
)

from evaluation.report_generator import print_comparison_table
from evaluation.visualization import plot_metric_comparison

cfg  = Config()
cfg.ensure_dirs()
set_seed(cfg.seed)
qcfg = QuantConfig()

device = torch.device('cpu')   # PTQ targets CPU edge inference
print('Device:', device_info(device))

In [ ]:
# ── Load FP32 model ───────────────────────────────────────────────────────────
fp32_model = load_model(cfg, device=device)
fp32_model = prepare_model_for_export(fp32_model)
print('FP32 model loaded.')

In [ ]:
# ── Step 1: Export FP32 → ONNX ───────────────────────────────────────────────
fp32_onnx_path = export_to_onnx(
    fp32_model, cfg,
    filename='baseline_fp32.onnx',
    opset_version=qcfg.opset_version,
)
print(f'FP32 ONNX size : {onnx_size_mb(fp32_onnx_path):.2f} MB')

# Inspect ops (optional)
list_onnx_ops(fp32_onnx_path)

In [ ]:
# ── Step 2: FP32 ONNX benchmark ──────────────────────────────────────────────
print('Benchmarking FP32 ONNX ...')
fp32_session   = create_ort_session(fp32_onnx_path)
fp32_input     = fp32_session.get_inputs()[0].name

fp32_latency   = benchmark_ort_session(fp32_session, cfg, input_name=fp32_input)
fp32_accuracy  = evaluate_ort_model(fp32_session, cfg, input_name=fp32_input)

print(f"FP32 Mean latency : {fp32_latency['mean_ms']:.2f} ms")
print(f"FP32 FPS          : {fp32_latency['fps']:.1f}")
print(f"FP32 Mean Dice    : {fp32_accuracy['mean_dice']:.4f}")

In [ ]:
# ── Step 3: Static INT8 Quantization ─────────────────────────────────────────
from quantization import quantize_static_onnx

int8_static_path = fp32_onnx_path.replace('fp32', 'int8_static')
print('Applying static INT8 quantization (with calibration) ...')

int8_static_path = quantize_static_onnx(
    fp32_onnx_path, int8_static_path, cfg, qcfg
)
print(f'INT8 static size : {onnx_size_mb(int8_static_path):.2f} MB')
print(f'Compression ratio: {compression_ratio(onnx_size_mb(fp32_onnx_path), onnx_size_mb(int8_static_path)):.2f}×')

In [ ]:
# ── Step 4: Static INT8 benchmark ────────────────────────────────────────────
int8_session  = create_ort_session(int8_static_path)
int8_input    = int8_session.get_inputs()[0].name

int8_latency  = benchmark_ort_session(int8_session, cfg, input_name=int8_input)
int8_accuracy = evaluate_ort_model(int8_session, cfg, input_name=int8_input)

print(f"INT8 Mean latency : {int8_latency['mean_ms']:.2f} ms")
print(f"INT8 FPS          : {int8_latency['fps']:.1f}")
print(f"INT8 Mean Dice    : {int8_accuracy['mean_dice']:.4f}")
speedup = fp32_latency['mean_ms'] / int8_latency['mean_ms']
print(f"Speedup           : {speedup:.2f}×")

In [ ]:
# ── Step 5: Per-class Dice comparison ────────────────────────────────────────
print('\n── Per-Class Dice: FP32 vs INT8 ─────────────────────────')
print(f'{"Class":<20} {"FP32":>8} {"INT8":>8} {"Drop":>8}')
print('-' * 48)
for name in CLASS_NAMES:
    key  = f"dice_{name.lower().replace(' ', '_')}"
    fp32 = fp32_accuracy.get(key, 0.0)
    int8 = int8_accuracy.get(key, 0.0)
    drop = fp32 - int8
    print(f'{name:<20} {fp32:>8.4f} {int8:>8.4f} {drop:>+8.4f}')

In [ ]:
# ── Step 6: Comparison table & plots ─────────────────────────────────────────
comparison = [
    {
        'label':    'FP32 ONNX',
        'mean_dice': fp32_accuracy['mean_dice'],
        'fps':       fp32_latency['fps'],
        'mean_ms':   fp32_latency['mean_ms'],
        'model_size_mb': onnx_size_mb(fp32_onnx_path),
        'energy_mj_per_inference': 0.0,
    },
    {
        'label':    'INT8 Static',
        'mean_dice': int8_accuracy['mean_dice'],
        'fps':       int8_latency['fps'],
        'mean_ms':   int8_latency['mean_ms'],
        'model_size_mb': onnx_size_mb(int8_static_path),
        'energy_mj_per_inference': 0.0,
    },
]
print_comparison_table(comparison)

plot_metric_comparison(
    labels=['FP32 ONNX', 'INT8 Static'],
    metrics={
        'Mean Dice': [fp32_accuracy['mean_dice'], int8_accuracy['mean_dice']],
        'Mean IoU':  [fp32_accuracy['mean_iou'],  int8_accuracy['mean_iou']],
        'Pixel Acc': [fp32_accuracy['pixel_acc'], int8_accuracy['pixel_acc']],
    },
    cfg=cfg,
    filename='ptq_accuracy_comparison.png',
    show=True,
)